# 14 - Contextual chunk prefixes: the document side of the vocabulary gap

> **Run order.** This notebook is step 14 of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` - the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.

[ADR-007](../docs/adr/0007-retrieval-strategy.md) closed with a prediction:

> The answer element carries no company name, no fiscal year, and no statement
> title - a chunk prefix of `SUNPHARMA FY2024 - Statement of Profit and Loss`
> would give both halves of the retriever something to match.

Query expansion fixed the **question** side of the vocabulary gap. This notebook
tests the **document** side. Two thirds of that prediction survived measurement;
one third did not, and finding out which is the point of the first section.

## 1. What the chunks actually carry

Before building anything: `heading` is already prepended to every chunk, so the
prefix only earns its place for context that is genuinely missing.

In [1]:
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

from analyst import evaluation as ev
from analyst.config import get_settings
from analyst.indexing import load_chunks

settings = get_settings()
questions = ev.load_questions(settings.data_dir / "benchmark" / "questions.jsonl")
print(f"{len(questions)} questions   benchmark {ev.bench_sha(questions)}")

base = load_chunks(with_context=False)
print(f"{len(base):,} chunks")

# Which of ADR-007's three claims hold?
have_heading = sum(1 for c in base if c.heading)
have_ticker = sum(1 for c in base if c.ticker.lower() in c.text.lower())
have_fy = sum(1 for c in base if str(c.fiscal_year) in c.text)
pd.DataFrame([
    {"context": "a heading", "chunks": have_heading, "share": have_heading / len(base)},
    {"context": "its own company", "chunks": have_ticker, "share": have_ticker / len(base)},
    {"context": "its fiscal year", "chunks": have_fy, "share": have_fy / len(base)},
]).assign(share=lambda d: (d["share"] * 100).round(1))

44 questions   benchmark 2c4aedf3dcb75f7e

10,118 chunks

,context,chunks,share
0,a heading,10112,99.9
1,its own company,845,8.4
2,its fiscal year,3807,37.6


**The company is missing, the year mostly is not, and a heading is nearly always
present.** So the interesting question is what those headings *say*.

In [2]:
from collections import Counter

heads = Counter(c.heading for c in base if c.heading)
reuse = pd.Series(list(heads.values()))
print(f"distinct headings: {len(heads):,}   reuse p50={reuse.median():.0f} "
      f"p90={reuse.quantile(0.9):.0f} max={reuse.max()}")
print()
for h, n in heads.most_common(8):
    flat = " / ".join(h.splitlines())
    print(f"{n:>5}x  {flat[:76]}")

distinct headings: 2,290   reuse p50=2 p90=6 max=511

  511x  CONSOLIDATED FINANCIAL STATEMENTS OF ICICI BANK LIMITED

  382x  STANDALONE FINANCIAL STATEMENTS OF ICICI BANK LIMITED

  380x  FINANCIAL STATEMENTS OF ICICI BANK LIMITED / SCHEDULES

  341x  BOARD’S REPORT

  339x  Notes to the Consolidated Financial Statements

  274x  Notes to the Standalone Financial Statements

  250x  Notes to the Standalone Financial Statements for the year ended 31st March, 

  182x  INDEPENDENT AUDITOR’S REPORT (Contd.)

### These are running page bands, not section titles

`226 / Statutory Reports / Corporate Overview / Financial Statements` is printed
on every page of a section. One heading repeats on **507 chunks**. A string
identical across hundreds of chunks cannot help tell them apart - it only
dilutes the vector and spends the encoder's 512-token budget.

`analyst.chunking.clean_heading` drops them, matching **whole lines only**, so
`CONSOLIDATED FINANCIAL STATEMENTS OF ICICI BANK LIMITED` - which names the
company - survives while a bare `Financial Statements` band does not.

In [3]:
from analyst.chunking import clean_heading

for h in ["226\nStatutory Reports\nCorporate Overview\nFinancial Statements",
          "CONSOLIDATED FINANCIAL STATEMENTS OF ICICI BANK LIMITED",
          "212\nConsolidated Balance Sheet\nStatutory Reports",
          "Integrated Annual Report 2024-25\n53"]:
    print(f"{' / '.join(h.splitlines())[:58]:<60} ->  {clean_heading(h)}")

226 / Statutory Reports / Corporate Overview / Financial S   ->  None

CONSOLIDATED FINANCIAL STATEMENTS OF ICICI BANK LIMITED      ->  CONSOLIDATED FINANCIAL STATEMENTS OF ICICI BANK LIMITED

212 / Consolidated Balance Sheet / Statutory Reports         ->  Consolidated Balance Sheet

Integrated Annual Report 2024-25 / 53                        ->  None

## 2. The part of ADR-007's prediction that did not survive

The prediction wanted a **statement title** in the prefix. Statement titles do
exist in the elements, so the obvious move is to scan backwards for the nearest
one. That was measured before it was built - and it does not work.

In [4]:
import re

from sqlalchemy import select

from analyst.db import session_scope
from analyst.models import Document, ElementRow

TITLE = re.compile(
    r"(statement of profit and loss|balance sheet|cash flow statement"
    r"|statement of cash flows|profit and loss account|statement of changes in equity)",
    re.IGNORECASE,
)

with session_scope() as s:
    order = {}
    for d in s.execute(select(Document)).scalars().all():
        rows = s.execute(
            select(ElementRow.element_id, ElementRow.page, ElementRow.text)
            .where(ElementRow.document_id == d.document_id)
            .order_by(ElementRow.page, ElementRow.seq)).all()
        order[d.document_id] = [(r[0], r[1], r[2] or "") for r in rows]

pos = {eid: (doc, i) for doc, els in order.items() for i, (eid, _, _) in enumerate(els)}

rows = []
for q in questions:
    for eid in q.expected_element_ids:
        if eid not in pos:
            continue
        doc, i = pos[eid]
        els = order[doc]
        back, title = None, ""
        for j in range(i, max(-1, i - 60), -1):
            if TITLE.search(els[j][2][:150]):
                back, title = i - j, " ".join(els[j][2][:80].split())
                break
        rows.append({"back": back, "same_page": back is not None and els[i - back][1] == els[i][1],
                     "title": title})

t = pd.DataFrame(rows)
found = t["back"].notna()
print(f"answer elements: {len(t)}")
print(f"a statement title within 60 elements : {found.sum()} ({found.mean():.1%})")
print(f"   ... on the element's own page     : {t.loc[found, 'same_page'].mean():.1%}")
print(f"   ... median distance back          : {t.loc[found, 'back'].median():.0f} elements")
print("\nwhat the scan actually recovers:")
for x, n in Counter(t.loc[found, "title"]).most_common(6):
    print(f"   {n:>3}x  {x[:74]}")

answer elements: 54

a statement title within 60 elements : 35 (64.8%)

   ... on the element's own page     : 2.9%

   ... median distance back          : 21 elements


what the scan actually recovers:

...

     5x  Consolidated Balance Sheet

     5x  Consolidated Statement of Changes in Equity

     4x  The Schedules referred to above form an integral part of the Consolidated 

     4x  forming part of the Consolidated Balance Sheet

     2x  Consolidated Balance Sheet As at 31st March, 2025

**Rejected.** A title is found for only ~65% of answer elements, almost never on
the element's own page, a median of 21 elements back - and most matches are
*prose mentions*, not titles: "Refer consolidated statement of changes in equity
for detailed movement...". Attaching those would label chunks with confident,
wrong context more often than right context.

So the prefix carries **only what is derivable with certainty**: the company and
the year, in both the vocabulary the question uses and the vocabulary the filing
prints. `analyst.chunking.DocContext`.

In [5]:
from analyst.chunking import DocContext

print(DocContext(ticker="SUNPHARMA", company="Sun Pharmaceutical Industries",
                 fiscal_year=2024).prefix)

Sun Pharmaceutical Industries (SUNPHARMA) FY2024 year ended March 31, 2024

## 3. A bug found on the way: chunks still over the encoder budget

A prefix is paid for out of the same 512 tokens as the passage, so the budget had
to be checked before adding to it. It was not being kept.

Day 3 capped the **row-packing** path at `MAX_TABLE_CHARS`. Three other paths had
no cap at all: a table whose JSON has no rows, a single text element larger than
the whole target, and - the expensive one - a table whose **header** is itself
oversized, since the header is repeated on every slice.

In [6]:
from analyst.chunking import budget

over = [c for c in base if len(c.embed_text) > budget(c.type)]
print(f"chunks over the encoder budget: {len(over)}")
print("  (this notebook builds `base` AFTER the fix, so it reads 0 here)")

# The worst offender in the corpus, before the fix: a 3,328-character "header".
with session_scope() as s:
    tj = s.execute(select(ElementRow.table_json).where(
        ElementRow.element_id == "HDFCBANK-annual_report-FY2025-2a1879ee:p0052:e0000")).scalar()
from analyst.chunking import _table_parts
hdr, rws = _table_parts(tj)
print(f"\nworst table: {len(hdr)} header cells totalling "
      f"{len(' | '.join(hdr)):,} chars, over {len(rws)} rows")
print("A header that leaves no room for a row is not a header - it is content,")
print("and repeating it on every slice put 138 chunks past the encoder limit.")

chunks over the encoder budget: 0

  (this notebook builds `base` AFTER the fix, so it reads 0 here)


worst table: 3 header cells totalling 3,328 chars, over 6 rows

A header that leaves no room for a row is not a header - it is content,

and repeating it on every slice put 138 chunks past the encoder limit.

## 4. Build the arms

Three things changed at once, so the experiment has to separate them:

| arm | budget fix | furniture stripped | context prefix |
|---|---|---|---|
| `fix` | yes | no | no |
| `ctx` | yes | yes | yes |
| `strip` | yes | yes | **no** |

The old `elements_bge-small` collection is none of these - it predates the budget
fix. Comparing `ctx` straight against it would credit the prefix with
un-truncating 136 chunks, so `fix` exists to hold that constant.

**`strip` was added after the first run.** `ctx` moved R@5 by +0.25, but it
changes *two* things at once, so that number could not be assigned to either.
`strip` removes the furniture and adds no prefix, which splits the difference in
two: `fix -> strip` is what stripping is worth, `strip -> ctx` is what the prefix
is worth on top of it. It is dense-only - the attribution question is answered
without paying for a second hybrid index.

In [7]:
ctx_chunks = load_chunks(with_context=True)
strip_chunks = load_chunks(with_context=False, strip_furniture=True)

# variant, chunks, index a hybrid collection too?
ARMS = [("fix", base, True), ("ctx", ctx_chunks, True), ("strip", strip_chunks, False)]

rows = []
for label, cs, _hy in ARMS:
    rows.append({
        "arm": label, "chunks": len(cs),
        "with heading": sum(1 for c in cs if c.heading),
        "over budget": sum(1 for c in cs if len(c.embed_text) > budget(c.type)),
        "median chars": int(pd.Series([len(c.embed_text) for c in cs]).median()),
        "max chars": max(len(c.embed_text) for c in cs),
    })
pd.DataFrame(rows)

,arm,chunks,with heading,over budget,median chars,max chars
0,fix,10118,10112,0,830,2046
1,ctx,10181,8141,0,862,2047
2,strip,10095,8080,0,821,2046


In [8]:
# What the encoder now reads for a chunk that answers a benchmark question.
sun = next(q for q in questions if q.ticker == "SUNPHARMA" and q.concept == "Total Revenue")
want = set(sun.expected_element_ids)
hit = next(c for c in ctx_chunks if want & set(c.element_ids))

print("QUESTION :", sun.question)
print("\nEMBEDDED :")
print(hit.embed_text[:300].replace("\n", " | "))
print("\nSTORED (what a citation quotes):")
print(hit.text[:160].replace("\n", " | "))

QUESTION :

What was Sun Pharmaceutical Industries's total revenue in FY2024?


EMBEDDED :

Sun Pharmaceutical Industries (SUNPHARMA) FY2024 year ended March 31, 2024 |  | Year ended | March 31, 2024 | Year ended | March 31, 2023 | Revenue from contracts with customers (Refer note 53) | 477,584.5 | 432,788.7 | Other operating revenues* | 7,384.0 | 6,068.1 |  | 484,968.5 | 438,856.8


STORED (what a citation quotes):

 | Year ended | March 31, 2024 | Year ended | March 31, 2023 | Revenue from contracts with customers (Refer note 53) | 477,584.5 | 432,788.7 | Other operating revenues*

`embed_text` and `text` are deliberately different. The prefix is something we
assembled; quoting it back as though the filing said it would be a provenance
bug, so `text` stays verbatim and only the embedding sees the prefix.

## 5. Index

Four collections: both arms, dense and hybrid. ~30 minutes each - **run this
unattended.** Resumable: a complete collection is skipped, so a failure part way
through does not cost the whole sweep.

In [9]:
import time

from analyst.retrievers import open_hybrid, open_store

MODEL = "bge-small"
BATCH = 256
FORCE = False

rows = []
for variant, cs, do_hybrid in ARMS:
    # --- dense
    embedder, store = open_store(settings, MODEL, variant)
    if FORCE or not store.exists() or store.count() != len(cs):
        store.recreate()
        t0 = time.perf_counter()
        for i in range(0, len(cs), BATCH):
            w = cs[i : i + BATCH]
            store.upsert(w, list(embedder.embed_documents([c.embed_text for c in w])))
        rows.append({"collection": store.collection, "points": store.count(),
                     "minutes": round((time.perf_counter() - t0) / 60, 1)})
    print(f"{store.collection:<28} {store.count():>6,} points")

    if not do_hybrid:
        continue

    # --- hybrid (its own collection: Qdrant fixes vector layout at creation)
    embedder, sparse, hstore = open_hybrid(settings, MODEL, variant)
    if FORCE or not hstore.exists() or hstore.count() != len(cs):
        hstore.recreate()
        t0 = time.perf_counter()
        for i in range(0, len(cs), BATCH):
            w = cs[i : i + BATCH]
            texts = [c.embed_text for c in w]
            hstore.upsert(w, list(embedder.embed_documents(texts)),
                          list(sparse.embed_documents(texts)))
        rows.append({"collection": hstore.collection, "points": hstore.count(),
                     "minutes": round((time.perf_counter() - t0) / 60, 1)})
    print(f"{hstore.collection:<28} {hstore.count():>6,} points")

pd.DataFrame(rows) if rows else "all four collections already complete"

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


elements_fix_bge-small       10,118 points

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


elements_hybrid_fix_bge-small 10,118 points

elements_ctx_bge-small       10,181 points

elements_hybrid_ctx_bge-small 10,181 points

elements_strip_bge-small     10,095 points

,collection,points,minutes
0,elements_strip_bge-small,10095,28.9


## 6. Measure

Same 44 questions, same injected-retriever scoring, same ledger. Query expansion
is on everywhere - ADR-007 adopted it, so it is the baseline this builds on, not
a variable.

In [10]:
from analyst.retrievers import dense, hybrid


def record(label: str, search, points: int, notes: str) -> ev.Run:
    cfg = ev.RunConfig(retriever=label, model=MODEL, filters="ticker+year",
                       limit=max(ev.K_VALUES), points=points, notes=notes)
    run = ev.build_run(
        cfg,
        ev.evaluate(questions, search, limit=max(ev.K_VALUES)),
        questions,
        deep=ev.evaluate(questions, search, limit=max(ev.DEPTHS)),
        root=ev.ROOT,
    )
    ev.append_run(run)
    print(f"{label:<24} R@5 {run.metrics.recall_at[5]:.3f}   "
          f"ceiling@200 {run.depth_curve.get(200, 0):.3f}")
    return run


NOTES = {"fix": "ADR-008 encoder-budget fix only",
         "ctx": "ADR-008 furniture stripped + context prefix",
         "strip": "ADR-008 furniture stripped, NO prefix - attribution arm"}

results = {}
for variant, _cs, do_hybrid in ARMS:
    embedder, store = open_store(settings, MODEL, variant)
    results[f"dense+expand[{variant}]"] = record(
        f"dense+expand[{variant}]",
        dense(embedder, store, "ticker+year", expand=True), store.count(), NOTES[variant])
    if not do_hybrid:
        continue
    _, sparse, hstore = open_hybrid(settings, MODEL, variant)
    results[f"hybrid+expand[{variant}]"] = record(
        f"hybrid+expand[{variant}]",
        hybrid(embedder, sparse, hstore, "ticker+year", expand=True),
        hstore.count(), NOTES[variant])

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


dense+expand[fix]        R@5 0.068   ceiling@200 0.682

hybrid+expand[fix]       R@5 0.068   ceiling@200 0.727

dense+expand[ctx]        R@5 0.318   ceiling@200 1.000

hybrid+expand[ctx]       R@5 0.341   ceiling@200 0.977

dense+expand[strip]      R@5 0.045   ceiling@200 0.727

### Against ADR-007

Read the **ceiling** column, not R@5. On 44 questions shallow recall moves in
steps of 0.023 - one question - which is far too coarse to judge a change by.
Recall at depth 200 is whether the right evidence is being surfaced at all.

In [11]:
ledger = ev.load_runs()
prior = {r.config.retriever: r for r in ledger
         if r.config.retriever in ("dense+expand", "hybrid+expand")
         and r.config.model == MODEL and r.config.filters == "ticker+year"}

wanted = ["dense+expand", "dense+expand[fix]", "dense+expand[strip]", "dense+expand[ctx]",
          "hybrid+expand", "hybrid+expand[fix]", "hybrid+expand[ctx]"]
picked = {**prior, **results}
table = pd.DataFrame([picked[k].row() for k in wanted if k in picked])
print(table.to_string(index=False))

print()
curves = pd.DataFrame({k: picked[k].depth_curve for k in wanted if k in picked}).T
print(curves.rename_axis("retriever").to_string())

                                   run           retriever     model     filters    R@1    R@3    R@5   R@10    MRR   pR@5  points  p50_ms    bench           git
       dense+expand-bge-small-b60d0b39        dense+expand bge-small ticker+year 0.0455 0.0455 0.0682 0.1136 0.0559 0.1364    9982    87.7 2c4aedf3 2f3c9a3-dirty
  dense+expand[fix]-bge-small-21c3af53   dense+expand[fix] bge-small ticker+year 0.0455 0.0455 0.0682 0.1136 0.0559 0.1364   10118    89.7 2c4aedf3 a715497-dirty
dense+expand[strip]-bge-small-cc4276b5 dense+expand[strip] bge-small ticker+year 0.0455 0.0455 0.0455 0.0909 0.0503 0.1136   10095    89.5 2c4aedf3 a715497-dirty
  dense+expand[ctx]-bge-small-f1b65882   dense+expand[ctx] bge-small ticker+year 0.1818 0.2273 0.3182 0.4545 0.2428 0.3636   10181    91.7 2c4aedf3 a715497-dirty
      hybrid+expand-bge-small-6510b044       hybrid+expand bge-small ticker+year 0.0455 0.0455 0.0682 0.0909 0.0544 0.1136    9982    91.1 2c4aedf3 2f3c9a3-dirty
 hybrid+expand[fix]-bge-smal

                        1       5       10      20      50      100     200
retriever                                                                  
dense+expand         0.0455  0.0682  0.1136  0.1591  0.3636  0.4773  0.6818
dense+expand[fix]    0.0455  0.0682  0.1136  0.1591  0.3636  0.4545  0.6818
dense+expand[strip]  0.0455  0.0455  0.0909  0.2045  0.4091  0.5227  0.7273
dense+expand[ctx]    0.1818  0.3182  0.4545  0.6364  0.7955  0.8409  1.0000
hybrid+expand        0.0000  0.0455  0.1136  0.1591  0.4773  0.6136  0.7273
hybrid+expand[fix]   0.0000  0.0455  0.0909  0.1591  0.4545  0.6136  0.7273
hybrid+expand[ctx]   0.1591  0.3409  0.4545  0.6136  0.7955  0.8636  0.9773

### Splitting the credit

The whole reason `strip` exists. On the dense arm:

- **`fix` -> `strip`** is what removing the page furniture is worth on its own.
- **`strip` -> `ctx`** is what the context prefix adds on top of that.

If `strip` lands near `ctx`, the prefix is close to free and the real finding is
that query expansion had been amplifying the furniture. If it lands near `fix`,
the prefix is doing the work after all and the first run's reading was wrong.

In [12]:
d = {k: picked[k] for k in ("dense+expand[fix]", "dense+expand[strip]", "dense+expand[ctx]")
     if k in picked}
if len(d) == 3:
    f, s, c = (d[f"dense+expand[{v}]"] for v in ("fix", "strip", "ctx"))
    for label, a, b in (("furniture stripping (fix -> strip)", f, s),
                        ("context prefix    (strip -> ctx)", s, c)):
        print(f"{label:<36} "
              f"R@5 {b.metrics.recall_at[5] - a.metrics.recall_at[5]:+.3f}   "
              f"MRR {b.metrics.mrr - a.metrics.mrr:+.3f}   "
              f"@200 {b.depth_curve.get(200, 0) - a.depth_curve.get(200, 0):+.3f}")
else:
    print("run the strip arm first")

furniture stripping (fix -> strip)   R@5 -0.023   MRR -0.006   @200 +0.045

context prefix    (strip -> ctx)     R@5 +0.273   MRR +0.193   @200 +0.273

## 7. Where it still fails

In [13]:
embedder, store = open_store(settings, MODEL, "ctx")
_, sparse, hstore = open_hybrid(settings, MODEL, "ctx")
res = ev.evaluate(questions, hybrid(embedder, sparse, hstore, "ticker+year", expand=True),
                  limit=max(ev.DEPTHS))
df = pd.DataFrame([r.model_dump() for r in res])
print(df.groupby("question_type")["rank"].agg(n="size", found="count").to_string())
print()
print(df.groupby("ticker")["rank"].agg(n="size", found="count").to_string())

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


                n  found
question_type           
growth         10     10
value_lookup   34     33

            n  found
ticker              
HDFCBANK    2      2
ICICIBANK  10     10
RELIANCE    8      7
SUNPHARMA  24     24

## 8. The ledger

In [14]:
print(ev.write_leaderboard(ev.load_runs()))

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\results\leaderboard.md